# 🌿 CropVision — Week 1: EDA, Preprocessing & Augmentation
**Infotact DS/ML Internship | Project 2: Agriculture & Smart Farming**

---

### 📋 Week 1 Objectives
- Download and explore the **PlantVillage** dataset
- Perform **Exploratory Data Analysis (EDA)** — class distribution, sample images, pixel stats
- Build the **preprocessing pipeline** — resize, normalize
- Apply **data augmentation** — flips, rotations, brightness/contrast
- Create a **stratified train/val/test split** with no data leakage
- Build **TensorFlow tf.data datasets** ready for Week 2 model training

### ⚠️ Commit Reminder
Commit **3–5 times today**. Suggested messages:
```
eda: class distribution plotted, X imbalanced classes found
eda: pixel intensity analysis complete, ImageNet normalization confirmed
data-clean: resize + normalize pipeline built
augment: flip/rotate/brightness transforms verified
data-clean: stratified 70/15/15 split, no leakage confirmed
```

---
## 0. Install & Import Libraries

In [ ]:
# Run this once to install all dependencies
# After installing, you do NOT need to run this cell again
!pip install tensorflow opencv-python pillow matplotlib seaborn scikit-learn kaggle --quiet

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image, ImageEnhance
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Plotting style
plt.rcParams['figure.facecolor'] = '#f9f9f9'
plt.rcParams['axes.facecolor']   = '#ffffff'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

print('✅ All libraries imported successfully')
print(f'   NumPy     : {np.__version__}')

import tensorflow as tf
print(f'   TensorFlow: {tf.__version__}')

---
## 1. Dataset Download & Setup

We use the **PlantVillage** dataset from Kaggle.  
It contains **50,000+ labeled leaf images** across **38 plant disease classes**.

**Folder structure after unzipping:**
```
data/PlantVillage/
    Apple___Apple_scab/
        image001.jpg
        image002.jpg ...
    Apple___healthy/
        image001.jpg ...
    Tomato___Late_blight/
        ...
```
Each folder name = the class label.

In [ ]:
# ---------------------------------------------------------
# OPTION A: Download via Kaggle CLI (recommended)
# ---------------------------------------------------------
# Step 1: Go to https://www.kaggle.com/settings → API → Create New Token
# Step 2: Place the downloaded kaggle.json in ~/.kaggle/
# Step 3: Run the cells below

import os
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Uncomment and run this line to download:
# !kaggle datasets download -d emmarex/plantdisease -p data/
# !unzip -q data/plantdisease.zip -d data/

# ---------------------------------------------------------
# OPTION B: Manual download
# ---------------------------------------------------------
# 1. Go to: https://www.kaggle.com/datasets/emmarex/plantdisease
# 2. Click Download
# 3. Unzip into:  data/PlantVillage/

print('📁 Directories created: data/, outputs/')
print('⬇️  Download PlantVillage if not already done')

In [ ]:
# ─────────────────────────────────────────────
# CONFIG — update DATA_DIR if your path differs
# ─────────────────────────────────────────────
DATA_DIR    = Path('data/PlantVillage')   # ← update if needed
IMG_SIZE    = (224, 224)                  # standard input size for ResNet/MobileNet
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Verify the dataset folder exists
if DATA_DIR.exists():
    class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
    print(f'✅ Dataset found at: {DATA_DIR}')
    print(f'   Total classes: {len(class_names)}')
else:
    print(f'❌ Dataset NOT found at: {DATA_DIR}')
    print('   Please download PlantVillage first (see cell above)')

---
## 2. Load Dataset — Collect All Image Paths & Labels

In [ ]:
def load_dataset_info(data_dir):
    """
    Walk the PlantVillage directory.
    Returns:
        image_paths : list of Path objects
        labels      : list of string class names
        class_names : sorted list of unique classes
    """
    image_paths = []
    labels      = []
    class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])

    for class_name in class_names:
        class_dir = data_dir / class_name
        images = (
            list(class_dir.glob('*.jpg'))  +
            list(class_dir.glob('*.JPG'))  +
            list(class_dir.glob('*.png'))  +
            list(class_dir.glob('*.PNG'))
        )
        image_paths.extend(images)
        labels.extend([class_name] * len(images))

    return image_paths, labels, class_names


image_paths, labels, class_names = load_dataset_info(DATA_DIR)

print(f'📦 Total images   : {len(image_paths)}')
print(f'🏷️  Total classes  : {len(class_names)}')
print(f'\nFirst 5 classes:')
for c in class_names[:5]:
    print(f'   {c}')

---
## 3. EDA — Class Distribution

**Why this matters:**  
Imbalanced classes are a real problem. If 90% of images are "healthy" and only 2% are a rare disease, the model will learn to always predict "healthy" and still get 90% accuracy — but be useless.  
We need to **spot imbalanced classes** early and decide how to handle them (oversample, class weights, etc.).

In [ ]:
# Count images per class
counts = Counter(labels)
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

names  = [c[0].replace('___', ' | ').replace('_', ' ') for c in sorted_counts]
values = [c[1] for c in sorted_counts]
mean_count = np.mean(values)

# Color bars: red = underrepresented, green = well represented
colors = ['#e74c3c' if v < mean_count * 0.5 else '#27ae60' for v in values]

fig, ax = plt.subplots(figsize=(18, 6))
bars = ax.bar(range(len(names)), values, color=colors, edgecolor='none', width=0.7)
ax.axhline(mean_count, color='#f39c12', linestyle='--', linewidth=1.5,
           label=f'Mean: {mean_count:.0f} images')

ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Class Distribution — PlantVillage Dataset\n'
             '🔴 Underrepresented (<50% of mean)   |   🟢 Well-represented',
             fontsize=13, pad=15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('outputs/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print(f'\n📊 CLASS DISTRIBUTION SUMMARY')
print(f'   Total images : {sum(values)}')
print(f'   Mean per class: {mean_count:.0f}')
print(f'   Max: {values[0]} ({sorted_counts[0][0]})')
print(f'   Min: {values[-1]} ({sorted_counts[-1][0]})')

underrep = [(n, v) for n, v in sorted_counts if v < mean_count * 0.5]
if underrep:
    print(f'\n   ⚠️  Underrepresented classes ({len(underrep)}):')
    for name, count in underrep:
        print(f'      {name}: {count} images')
else:
    print('\n   ✅ No severely underrepresented classes found')

### 📝 Observation — Class Distribution

> **Write your findings here after running the cell above:**
> - How many classes are underrepresented?
> - What is the max / min image count?
> - Will you need to handle class imbalance? How?

*Example: "3 classes had fewer than 200 images (below 50% of the 450-image mean). We will use class_weight in model training to compensate."*

---
## 4. EDA — Sample Image Visualization

**Why this matters:**  
Always look at your data! You need to see what healthy vs diseased leaves actually look like. This also helps you spot any corrupted or mislabelled images early.

In [ ]:
# Group images by class
class_to_paths = {}
for path, label in zip(image_paths, labels):
    class_to_paths.setdefault(label, []).append(path)

# Show 1 sample from each of the first 9 classes (3×3 grid)
n_show = min(9, len(class_names))
selected = list(class_to_paths.items())[:n_show]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Sample Images per Disease Class\n'
             '🟢 Green border = Healthy   |   🔴 Red border = Diseased',
             fontsize=14, y=1.01)

for idx, (class_name, paths) in enumerate(selected):
    ax = axes[idx // 3][idx % 3]

    # Pick a random image from this class
    img_path = random.choice(paths)
    img = Image.open(img_path).convert('RGB').resize((224, 224))
    ax.imshow(img)

    # Format label for display
    display_label = class_name.replace('___', '\n').replace('_', ' ')
    ax.set_title(display_label, fontsize=9, pad=6)
    ax.axis('off')

    # Border color: green = healthy, red = diseased
    border_color = '#27ae60' if 'healthy' in class_name.lower() else '#e74c3c'
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig('outputs/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/eda_sample_images.png')

---
## 5. EDA — Pixel Intensity Analysis

**Why this matters:**  
Before normalizing, we check the raw pixel distribution.  
- Are pixels spread across [0, 255]? → Standard normalization to [0, 1] is fine.
- Are they skewed? → We might need channel-wise normalization.
- For Transfer Learning (ResNet, MobileNet), we use **ImageNet normalization** (mean/std per channel).

In [ ]:
# Sample 200 random images for pixel statistics
SAMPLE_SIZE = 200
sampled_idx = np.random.choice(len(image_paths), SAMPLE_SIZE, replace=False)

r_means, g_means, b_means = [], [], []

for i in sampled_idx:
    img_arr = np.array(
        Image.open(image_paths[i]).convert('RGB').resize((224, 224)),
        dtype=np.float32
    ) / 255.0
    r_means.append(img_arr[:, :, 0].mean())
    g_means.append(img_arr[:, :, 1].mean())
    b_means.append(img_arr[:, :, 2].mean())

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Per-Channel Pixel Intensity Distribution (sample={SAMPLE_SIZE} images)',
             fontsize=13)

channel_data   = [r_means, g_means, b_means]
channel_names  = ['Red Channel', 'Green Channel', 'Blue Channel']
channel_colors = ['#e74c3c', '#27ae60', '#2980b9']

for ax, data, name, color in zip(axes, channel_data, channel_names, channel_colors):
    ax.hist(data, bins=30, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(np.mean(data), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean = {np.mean(data):.3f}')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Mean pixel value (normalized 0–1)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/eda_pixel_stats.png', dpi=150, bbox_inches='tight')
plt.show()

# Print stats
print('\n📐 CHANNEL STATISTICS (sampled images):')
print(f'   Red   — mean: {np.mean(r_means):.4f}   std: {np.std(r_means):.4f}')
print(f'   Green — mean: {np.mean(g_means):.4f}   std: {np.std(g_means):.4f}')
print(f'   Blue  — mean: {np.mean(b_means):.4f}   std: {np.std(b_means):.4f}')
print()
print('📌 ImageNet normalization stats (used for Transfer Learning):')
print('   mean = [0.485, 0.456, 0.406]')
print('   std  = [0.229, 0.224, 0.225]')
print('   → These will be applied when using ResNet50/MobileNet in Week 3')

### 📝 Observation — Pixel Statistics

> **Write your findings here:**
> - Are the channel means close to 0.5 (balanced) or skewed?
> - Green channel is typically highest for leaf images — did you observe this?
> - Does this confirm that dividing by 255 is a reasonable normalization?

*Example: "Green channel has the highest mean (0.412) as expected for leaf images. All channels are in the 0.3–0.5 range, confirming standard [0,1] normalization is appropriate. Will switch to ImageNet mean/std normalization for Week 3 Transfer Learning."*

---
## 6. EDA — Image Size Check

**Why this matters:**  
Raw images may come in different sizes. Neural networks need a fixed input size.  
We must resize everything to **224×224** — the standard for ResNet50/MobileNet.

In [ ]:
# Sample 300 images and check their raw dimensions
SAMPLE_SIZE = 300
sampled_idx = np.random.choice(len(image_paths), SAMPLE_SIZE, replace=False)

widths, heights = [], []
unique_sizes = Counter()

for i in sampled_idx:
    with Image.open(image_paths[i]) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
        unique_sizes[(w, h)] += 1

# Plot width and height distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Raw Image Size Distribution (sample={SAMPLE_SIZE})', fontsize=13)

axes[0].hist(widths,  bins=20, color='#8e44ad', alpha=0.8, edgecolor='none')
axes[0].set_title('Image Widths')
axes[0].set_xlabel('Width (pixels)')
axes[0].axvline(np.mean(widths), color='black', linestyle='--',
                label=f'Mean = {np.mean(widths):.0f}px')
axes[0].legend()

axes[1].hist(heights, bins=20, color='#16a085', alpha=0.8, edgecolor='none')
axes[1].set_title('Image Heights')
axes[1].set_xlabel('Height (pixels)')
axes[1].axvline(np.mean(heights), color='black', linestyle='--',
                label=f'Mean = {np.mean(heights):.0f}px')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_image_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📏 IMAGE SIZE REPORT:')
print(f'   Unique sizes found: {len(unique_sizes)}')
print(f'   Most common sizes:')
for size, count in unique_sizes.most_common(5):
    print(f'      {size[0]}×{size[1]}px — {count} images')

if len(unique_sizes) > 1:
    print('\n   ⚠️  Images are NOT uniform — we MUST resize to 224×224')
else:
    print('\n   ✅ All images are already the same size')
print('\n   → Resizing to 224×224 in the preprocessing pipeline (Section 7)')

---
## 7. Preprocessing Pipeline — Resize + Normalize

Every image goes through this before being fed to the model:
1. **Convert to RGB** — handles grayscale or RGBA images
2. **Resize to 224×224** — fixed input size for the neural network
3. **Normalize to [0.0, 1.0]** — divide by 255 so all pixel values are small floats

In [ ]:
def preprocess_image(image_path, target_size=(224, 224)):
    """
    Load and preprocess a single image.

    Steps:
        1. Open image and convert to RGB
        2. Resize to target_size using high-quality LANCZOS filter
        3. Convert to float32 numpy array
        4. Normalize pixel values from [0, 255] → [0.0, 1.0]

    Args:
        image_path  : Path to the image file
        target_size : Tuple (width, height), default (224, 224)

    Returns:
        numpy array of shape (224, 224, 3), dtype float32
    """
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr


# ── Test the pipeline on one sample image ──
sample_path = image_paths[0]
processed   = preprocess_image(sample_path)

print('🖼️  Preprocessing test on one image:')
print(f'   Original file : {sample_path.name}')
with Image.open(sample_path) as img:
    print(f'   Original size : {img.size[0]}×{img.size[1]}px')
print(f'   Output shape  : {processed.shape}  ← (H, W, C)')
print(f'   Pixel range   : [{processed.min():.3f}, {processed.max():.3f}]')
print(f'   Mean value    : {processed.mean():.3f}')
print(f'   dtype         : {processed.dtype}')

# Visual: original vs processed side by side
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle('Preprocessing: Before vs After', fontsize=12)

original_display = Image.open(sample_path).convert('RGB')
axes[0].imshow(original_display)
axes[0].set_title(f'Original\n{original_display.size[0]}×{original_display.size[1]}px', fontsize=10)
axes[0].axis('off')

axes[1].imshow(processed)
axes[1].set_title(f'Preprocessed\n224×224px | values [0,1]', fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('outputs/preprocessing_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Data Augmentation

**Why this matters:**  
In the real world, a farmer might take a photo of the same leaf at different angles, in different lighting, or with shadows.  
Augmentation artificially creates these variations during training so the model is **robust to real-world conditions**.

⚠️ **Important:** Augmentation is applied to the **training set ONLY**.  
Validation and test sets always get the original, unmodified images — otherwise your metrics would be unreliable.

In [ ]:
def augment_image(pil_img, seed=None):
    """
    Apply random augmentations to a PIL Image.
    Used ONLY on training images.

    Transforms applied:
        - Horizontal flip   (50% chance) — leaf diseases appear on both sides
        - Vertical flip     (30% chance) — adds more variety
        - Random rotation   (±20°)       — different camera angles in the field
        - Brightness jitter (±30%)       — varying sunlight conditions
        - Contrast jitter   (±20%)       — shadow and lighting variation
        - Saturation jitter (±20%)       — color variation between devices

    Args:
        pil_img : PIL Image object
        seed    : optional int for reproducibility

    Returns:
        Augmented PIL Image object
    """
    if seed is not None:
        random.seed(seed)

    # 1. Horizontal flip
    if random.random() > 0.5:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)

    # 2. Vertical flip
    if random.random() > 0.7:
        pil_img = pil_img.transpose(Image.FLIP_TOP_BOTTOM)

    # 3. Random rotation ±20 degrees
    angle = random.uniform(-20, 20)
    pil_img = pil_img.rotate(angle, fillcolor=(0, 0, 0))

    # 4. Brightness jitter
    factor = random.uniform(0.7, 1.3)
    pil_img = ImageEnhance.Brightness(pil_img).enhance(factor)

    # 5. Contrast jitter
    factor = random.uniform(0.8, 1.2)
    pil_img = ImageEnhance.Contrast(pil_img).enhance(factor)

    # 6. Saturation jitter
    factor = random.uniform(0.8, 1.2)
    pil_img = ImageEnhance.Color(pil_img).enhance(factor)

    return pil_img


print('✅ augment_image() function defined')

In [ ]:
# Visualize augmentations — show original + 6 augmented versions
sample_path = image_paths[10]   # pick any image
original    = Image.open(sample_path).convert('RGB').resize((224, 224))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Data Augmentation Preview — Training Set Only\n'
             'Each version shows a different random transform combination',
             fontsize=13, y=1.01)

# Row 0: Original + 3 augmented
# Row 1: 4 more augmented
axes[0][0].imshow(original)
axes[0][0].set_title('Original', fontsize=10, color='#27ae60', fontweight='bold')
axes[0][0].axis('off')

aug_count = 0
for row in range(2):
    for col in range(4):
        if row == 0 and col == 0:
            continue  # skip — original already placed
        augmented = augment_image(original.copy(), seed=aug_count * 13)
        axes[row][col].imshow(augmented)
        axes[row][col].set_title(f'Augmented #{aug_count + 1}', fontsize=9, color='#8e44ad')
        axes[row][col].axis('off')
        aug_count += 1

plt.tight_layout()
plt.savefig('outputs/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/augmentation_preview.png')
print()
print('✅ Verify: all augmented images still look like realistic leaf photos')
print('   If any look too distorted or unrecognizable, reduce the augmentation range')

### 📝 Observation — Augmentation

> **Write your findings here:**
> - Do the augmented images still look like realistic leaf photos?
> - Is the rotation range (±20°) appropriate or too aggressive?
> - Does the brightness variation look realistic?

*Example: "All augmented images look realistic. The ±20° rotation is slightly aggressive for some images — the black fill corners are visible. Will consider reducing to ±15° or using reflect padding instead."*

---
## 9. Stratified Train / Validation / Test Split

**Why stratified?**  
A simple random split might accidentally put all images of a rare disease class into the test set.  
Stratified splitting ensures **every split has the same proportion** of each class.

**Split ratios:** 70% train / 15% val / 15% test

**Why separate validation and test?**
- **Validation set** → used during training to tune hyperparameters and monitor overfitting
- **Test set** → used ONCE at the very end to report final performance (never tune on this!)

In [ ]:
def stratified_split(data_dir, train_r=0.70, val_r=0.15, test_r=0.15, seed=42):
    """
    Split dataset into train/val/test sets stratified by class.

    Stratified means: each split contains the same
    proportion of each class as the full dataset.

    Returns:
        splits      : dict with keys 'train', 'val', 'test'
                      each value is a list of (path, label) tuples
        class_names : sorted list of class name strings
    """
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6, 'Ratios must sum to 1.0'
    random.seed(seed)

    splits = {'train': [], 'val': [], 'test': []}
    class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])

    for class_name in class_names:
        class_dir = data_dir / class_name
        all_imgs  = (
            list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) +
            list(class_dir.glob('*.png')) + list(class_dir.glob('*.PNG'))
        )
        random.shuffle(all_imgs)
        n = len(all_imgs)

        n_train = int(n * train_r)
        n_val   = int(n * val_r)

        splits['train'].extend([(p, class_name) for p in all_imgs[:n_train]])
        splits['val'  ].extend([(p, class_name) for p in all_imgs[n_train:n_train + n_val]])
        splits['test' ].extend([(p, class_name) for p in all_imgs[n_train + n_val:]])

    return splits, class_names


# Run the split
splits, class_names = stratified_split(DATA_DIR)

print('✅ STRATIFIED SPLIT COMPLETE')
print(f'   Train : {len(splits["train"]):>6} images  ({len(splits["train"])/sum(len(v) for v in splits.values())*100:.0f}%)')
print(f'   Val   : {len(splits["val"  ]):>6} images  ({len(splits["val"]  )/sum(len(v) for v in splits.values())*100:.0f}%)')
print(f'   Test  : {len(splits["test" ]):>6} images  ({len(splits["test" ])/sum(len(v) for v in splits.values())*100:.0f}%)')

In [ ]:
# ── CRITICAL: Verify NO data leakage between splits ──
# Every image path must appear in exactly ONE split

train_paths = set(str(p) for p, _ in splits['train'])
val_paths   = set(str(p) for p, _ in splits['val'])
test_paths  = set(str(p) for p, _ in splits['test'])

tv_overlap  = len(train_paths & val_paths)
tt_overlap  = len(train_paths & test_paths)
vt_overlap  = len(val_paths   & test_paths)

print('🔍 DATA LEAKAGE CHECK:')
print(f'   Train ∩ Val  overlap: {tv_overlap} images')
print(f'   Train ∩ Test overlap: {tt_overlap} images')
print(f'   Val   ∩ Test overlap: {vt_overlap} images')

if tv_overlap == 0 and tt_overlap == 0 and vt_overlap == 0:
    print('\n   ✅ No data leakage detected! Splits are clean.')
else:
    print('\n   ❌ DATA LEAKAGE DETECTED! Fix the split logic before proceeding.')

In [ ]:
# Visualize the split distribution across classes
train_counts = Counter(label for _, label in splits['train'])
val_counts   = Counter(label for _, label in splits['val'])
test_counts  = Counter(label for _, label in splits['test'])

x = np.arange(len(class_names))
t_vals  = [train_counts.get(c, 0) for c in class_names]
v_vals  = [val_counts.get(c,   0) for c in class_names]
te_vals = [test_counts.get(c,  0) for c in class_names]

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(x, t_vals,  width=0.6, label='Train (70%)', color='#27ae60', edgecolor='none')
ax.bar(x, v_vals,  width=0.6, label='Val (15%)',   color='#f39c12',
       bottom=t_vals, edgecolor='none')
ax.bar(x, te_vals, width=0.6, label='Test (15%)',  color='#e74c3c',
       bottom=[t + v for t, v in zip(t_vals, v_vals)], edgecolor='none')

short_labels = [c.replace('___', '\n').replace('_', ' ') for c in class_names]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=90, fontsize=7)
ax.set_ylabel('Image count', fontsize=11)
ax.set_title('Stratified Train / Val / Test Split per Class', fontsize=13, pad=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('outputs/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/split_distribution.png')

---
## 10. Build TensorFlow tf.data Datasets

**Why tf.data?**  
Loading images one-by-one during training is slow.  
`tf.data.Dataset` pipelines are **fast and memory-efficient** — they load and augment images in parallel while the GPU trains on the previous batch.

This dataset object will be fed directly into `model.fit()` in Week 2.

In [ ]:
import tensorflow as tf

# Map class names to integer indices (0, 1, 2, ...)
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
NUM_CLASSES  = len(class_names)
BATCH_SIZE   = 32
IMG_SIZE_TF  = [224, 224]

print(f'Number of classes : {NUM_CLASSES}')
print(f'Batch size        : {BATCH_SIZE}')
print(f'Example mappings  :')
for name, idx in list(label_to_idx.items())[:4]:
    print(f'   {name} → {idx}')


# ── Step 1: Load and preprocess a single image (pure TF ops) ──
def load_and_preprocess(image_path, label_idx):
    raw = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, IMG_SIZE_TF)
    img = tf.cast(img, tf.float32) / 255.0      # normalize to [0, 1]
    label = tf.one_hot(label_idx, NUM_CLASSES)  # one-hot encode
    return img, label


# ── Step 2: Augmentation (TF ops — runs on GPU during training) ──
def augment_tf(img, label):
    img = tf.image.random_flip_left_right(img)       # horizontal flip
    img = tf.image.random_flip_up_down(img)          # vertical flip
    img = tf.image.random_brightness(img, 0.3)       # brightness jitter
    img = tf.image.random_contrast(img, 0.8, 1.2)    # contrast jitter
    img = tf.image.random_saturation(img, 0.8, 1.2)  # saturation jitter
    img = tf.clip_by_value(img, 0.0, 1.0)            # clamp to [0,1]
    return img, label


# ── Step 3: Build dataset from split lists ──
def make_dataset(split_data, shuffle=False, augment=False):
    """
    Create a tf.data.Dataset from a list of (path, label) tuples.

    Args:
        split_data : list of (Path, class_name) tuples
        shuffle    : True for training set only
        augment    : True for training set only
    """
    paths  = [str(p) for p, _ in split_data]
    labels = [label_to_idx[l] for _, l in split_data]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=RANDOM_SEED)

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


# Build all three datasets
train_ds = make_dataset(splits['train'], shuffle=True,  augment=True)
val_ds   = make_dataset(splits['val'],   shuffle=False, augment=False)
test_ds  = make_dataset(splits['test'],  shuffle=False, augment=False)

print('\n✅ TF Datasets built successfully:')
print(f'   train_ds : {len(train_ds)} batches of {BATCH_SIZE}')
print(f'   val_ds   : {len(val_ds)} batches of {BATCH_SIZE}')
print(f'   test_ds  : {len(test_ds)} batches of {BATCH_SIZE}')

In [ ]:
# Verify one batch — check shapes and value ranges
for img_batch, label_batch in train_ds.take(1):
    print('🔍 Verifying one training batch:')
    print(f'   Image batch shape : {img_batch.shape}  ← (batch, H, W, C)')
    print(f'   Label batch shape : {label_batch.shape} ← (batch, num_classes)')
    print(f'   Pixel range       : [{img_batch.numpy().min():.3f}, {img_batch.numpy().max():.3f}]')
    print(f'   dtype             : {img_batch.dtype}')

# Show a sample batch of 8 images
idx_to_label = {v: k for k, v in label_to_idx.items()}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Batch from train_ds (with augmentation)', fontsize=13)

for img_batch, label_batch in train_ds.take(1):
    for i, ax in enumerate(axes.flat):
        img = img_batch[i].numpy()
        lbl = idx_to_label[np.argmax(label_batch[i].numpy())]
        ax.imshow(img)
        ax.set_title(lbl.replace('___', '\n').replace('_', ' '), fontsize=8)
        ax.axis('off')

plt.tight_layout()
plt.savefig('outputs/sample_training_batch.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: outputs/sample_training_batch.png')

---
## 11. Save Class Names for Week 2

We save `class_names` to a JSON file so Week 2 and Week 3 notebooks can load it without re-running the split.

In [ ]:
# Save class names
with open('outputs/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

# Save label mapping
with open('outputs/label_to_idx.json', 'w') as f:
    json.dump(label_to_idx, f, indent=2)

print('💾 Saved: outputs/class_names.json')
print('💾 Saved: outputs/label_to_idx.json')
print(f'\n📋 Total classes saved: {len(class_names)}')
print('\nFirst 5 class names:')
for i, name in enumerate(class_names[:5]):
    print(f'   [{i}] {name}')

---
## 12. Week 1 Summary & Commit Checklist

### ✅ What we accomplished this week

| Task | Status |
|------|--------|
| Downloaded PlantVillage dataset | ✅ |
| EDA — class distribution plot | ✅ |
| EDA — sample images per class | ✅ |
| EDA — pixel intensity analysis | ✅ |
| EDA — image size consistency check | ✅ |
| Preprocessing pipeline (resize + normalize) | ✅ |
| Data augmentation (7 transforms) | ✅ |
| Stratified 70/15/15 split | ✅ |
| Data leakage verification | ✅ |
| TF datasets (train_ds, val_ds, test_ds) | ✅ |
| Saved class_names.json for Week 2 | ✅ |

### 📁 Outputs generated
```
outputs/
    eda_class_distribution.png
    eda_sample_images.png
    eda_pixel_stats.png
    eda_image_sizes.png
    preprocessing_before_after.png
    augmentation_preview.png
    split_distribution.png
    sample_training_batch.png
    class_names.json
    label_to_idx.json
```

### 🔜 Week 2 Preview
- Build a **custom CNN** from scratch (Conv2D → MaxPool → Dense)
- Train on `train_ds`, validate on `val_ds`
- Plot loss/accuracy curves
- Detect and fix **overfitting** using Dropout + Early Stopping

---

### 🔀 Commit these files now!
```bash
# Clear notebook outputs first!
# In Jupyter: Kernel → Restart & Clear Output
# Then save, then commit

git add Week1_CropVision_EDA_Preprocessing.ipynb outputs/
git commit -m 'eda: Week 1 complete — class distribution, pixel stats, preprocessing, augmentation, splits'
git push origin main
```